# Part F: Enseble model (Gradient boost and random forest)

## Stacking classfier 

In [15]:
import pandas as pd

train_df = pd.read_csv('credit_train_preprocessed.csv')
test_df = pd.read_csv('credit_test_preprocessed.csv')

# Separate features (X) and target (y)
X_train_resampled = train_df.drop(columns=['dpnm'])
y_train_resampled = train_df['dpnm']

X_test = test_df.drop(columns=['dpnm'])
y_test = test_df['dpnm']

print("Data loaded")

Data loaded


In [18]:

from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, f1_score
import joblib

#load optimised models from part E
best_rf = joblib.load('best_rf.joblib')
best_gb = joblib.load('best_gb.joblib')

#Define the best estimators
base_estimators = [
    ('rf', best_rf),
    ('gb', best_gb)
]

# initilise the stacking classifier 
stacking_ensemble = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(random_state=42),
    cv=5,
    n_jobs=-1
)

# training the ensemble on data
stacking_ensemble.fit(X_train_resampled, y_train_resampled)

# predict using stacked data
y_pred_stack = stacking_ensemble.predict(X_test)
y_prob_stack = stacking_ensemble.predict_proba(X_test)[:, 1]

print("Ensemble Performance")
print(f"Ensemble ROC-AUC: {roc_auc_score(y_test, y_prob_stack):.4f}")
print(f"Ensemble F1-Score: {f1_score(y_test, y_pred_stack):.4f}\n")

print(classification_report(y_test, y_pred_stack))

joblib.dump(stacking_ensemble, 'stacking_ensemble.joblib')

print("Stacking ensemble saved successfully!")


Ensemble Performance
Ensemble ROC-AUC: 0.7676
Ensemble F1-Score: 0.5034

              precision    recall  f1-score   support

           0       0.85      0.90      0.88      4667
           1       0.57      0.45      0.50      1326

    accuracy                           0.80      5993
   macro avg       0.71      0.68      0.69      5993
weighted avg       0.79      0.80      0.79      5993

Stacking ensemble saved successfully!
